# Head-office TPP split artifacts

이 노트북은 `sample_data/head_office/marked_target_df.parquet`를 quantity reconstruction 실험용 `train / validation / test` parquet로 분리합니다.

핵심 원칙은 다음과 같습니다.

- 각 `oper_part_no` 시계열 내부에서 chronological split을 수행합니다.
- magnitude `mark`와 `scale_residual`은 train split 기준으로 tail merge rule을 fit한 뒤 전체 split에 동일하게 적용합니다.
- full TTM 실험에서 leakage를 추적할 수 있도록 `chronological_split` 컬럼이 포함된 `with_split` 파일도 함께 저장합니다.

## 1. Setup

노트북 실행 위치가 project root이든 notebook 폴더이든 동일하게 동작하도록 preprocessing utility와 project root를 resolve합니다.

In [1]:
from pathlib import Path
import sys

def resolve_project_root(start: Path | None = None) -> Path:
    """Find the project root even when Jupyter starts from home or /tmp."""
    anchor = (start or Path.cwd()).expanduser().resolve()
    base = anchor if anchor.is_dir() else anchor.parent
    candidates = [base, *base.parents]
    candidates.extend([
        Path('~/workspace/paper_research').expanduser(),
        Path('/home/workspace/paper_research'),
        Path('/Users/igwanhyeong/PycharmProjects/paper_research'),
    ])
    for candidate in candidates:
        if all((candidate / name).exists() for name in ('sample_data', 'models', 'utils')):
            return candidate
    raise RuntimeError('Could not locate the paper_research project root.')

PROJECT_ROOT = resolve_project_root()
PREPROCESSING_DIR = PROJECT_ROOT / 'simple_lab_test' / 'notebooks' / 'preprocessing'
for path in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from tpp_split_utils import SplitConfig, build_and_save_quantity_splits, ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

PROJECT_ROOT = /home/leekwanhyeong/workspace/paper_research


## 2. Split configuration

Head-office 데이터는 수량 tail이 길기 때문에 기존 실험 정책과 동일하게 `scale_base=2.0`을 사용합니다. 출력 prefix는 외부 benchmark 스타일에 맞춰 `marked_target_*`로 둡니다.

In [2]:
cfg = SplitConfig(
    dataset_name='head_office',
    input_path=PROJECT_ROOT / 'sample_data' / 'head_office' / 'marked_target_df.parquet',
    output_dir=PROJECT_ROOT / 'sample_data' / 'head_office',
    output_prefix='marked_target',
    scale_base=2.0,
    train_ratio=0.70,
    validation_ratio=0.15,
    test_ratio=0.15,
)
cfg

SplitConfig(dataset_name='head_office', input_path=PosixPath('/home/leekwanhyeong/workspace/paper_research/sample_data/head_office/marked_target_df.parquet'), output_dir=PosixPath('/home/leekwanhyeong/workspace/paper_research/sample_data/head_office'), output_prefix='marked_target', scale_base=2.0, train_ratio=0.7, validation_ratio=0.15, test_ratio=0.15, entity_col='oper_part_no', order_col='seq', clip_min_qty=1.0, min_count=100, min_coverage=0.999)

## 3. Build and save artifacts

이 셀은 다음 파일을 생성합니다.

- `marked_target_with_split.parquet`
- `marked_target_train.parquet`
- `marked_target_validation.parquet`
- `marked_target_test.parquet`
- `marked_target_split_manifest.json`

In [3]:
result = build_and_save_quantity_splits(cfg, overwrite=True)
print('max_order fitted on train =', result['max_order'])
for name, path in result['paths'].items():
    print(f'{name}: {path}')

max_order fitted on train = 10
with_split: /home/leekwanhyeong/workspace/paper_research/sample_data/head_office/marked_target_with_split.parquet
train: /home/leekwanhyeong/workspace/paper_research/sample_data/head_office/marked_target_train.parquet
validation: /home/leekwanhyeong/workspace/paper_research/sample_data/head_office/marked_target_validation.parquet
test: /home/leekwanhyeong/workspace/paper_research/sample_data/head_office/marked_target_test.parquet
manifest: /home/leekwanhyeong/workspace/paper_research/sample_data/head_office/marked_target_split_manifest.json


## 4. Quality checks

저장 직후 split별 row 수, series 수, mark 분포를 확인합니다. `scale_residual`은 대부분 `[0, 1)`에 있어야 하지만, tail class로 clipping된 최상위 mark에서는 1 이상이 나올 수 있습니다.

In [4]:
from IPython.display import display
import polars as pl

labeled_df = result['labeled_df']
display(labeled_df.head())
display(pl.DataFrame(result['summary']['split_counts']))
display(pl.DataFrame(result['summary']['mark_counts']).pivot(values='len', index='mark', columns='chronological_split').fill_null(0).sort('mark'))
display(labeled_df.select([
    pl.len().alias('rows'),
    pl.col('oper_part_no').n_unique().alias('series'),
    pl.col('demand_qty').median().alias('qty_median'),
    pl.col('demand_qty').quantile(0.95).alias('qty_p95'),
    pl.col('demand_qty').max().alias('qty_max'),
    pl.col('scale_residual').min().alias('residual_min'),
    pl.col('scale_residual').max().alias('residual_max'),
]))

oper_part_no,demand_dt,seq,delta_t,demand_qty,chronological_split,log_qty,log10_qty,raw_order,mark,scale_residual,z
str,i64,i64,i32,f64,str,f64,f64,i32,i32,f64,f64
"""0001-1002""",201811,1,0,2.0,"""train""",1.0,0.30103,1,1,0.0,1.0
"""0001-1002""",201902,44,43,1.0,"""train""",0.0,0.0,0,0,0.0,0.0
"""0001-1002""",202309,260,216,1.0,"""train""",0.0,0.0,0,0,0.0,0.0
"""0001-1002""",202312,263,3,1.0,"""train""",0.0,0.0,0,0,0.0,0.0
"""0001-1002""",202609,416,153,1.0,"""validation""",0.0,0.0,0,0,0.0,0.0


chronological_split,rows,series,qty_median,qty_p95,qty_max
str,i64,i64,f64,f64,f64
"""test""",41344,16377,2.0,15.0,4000.0
"""train""",159643,23387,2.0,17.0,5000.0
"""validation""",41901,23387,2.0,15.0,5000.0


/tmp/ipykernel_1259636/3830830770.py:7: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  display(pl.DataFrame(result['summary']['mark_counts']).pivot(values='len', index='mark', columns='chronological_split').fill_null(0).sort('mark'))


mark,test,train,validation
i64,i64,i64,i64
0,17230,64403,17255
1,11954,47544,12115
2,6946,27024,7163
3,3253,12349,3428
4,1117,4771,1113
…,…,…,…
6,215,816,184
7,91,415,92
8,57,214,52


rows,series,qty_median,qty_p95,qty_max,residual_min,residual_max
u32,u32,f64,f64,f64,f64,f64
242888,23387,2.0,16.0,5000.0,0.0,2.287712
